## xAI (x.ai/news) Scraper

In [2]:
# [Updated] xAI Scraper using curl_cffi
try:
    from curl_cffi import requests
except ImportError:
    print("Error: curl_cffi not installed. Please run: pip install curl-cffi")
    import requests

from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import time
import re

# ==========================================
# xAI Scraper Helpers
# ==========================================

def parse_xai_date(date_str):
    """Parse dates like 'December 30, 2025'"""
    if not date_str: return None
    date_str = date_str.strip()
    formats = [
        '%B %d, %Y',       # December 30, 2025
        '%b %d, %Y',       # Dec 30, 2025
        '%Y-%m-%d',
    ]
    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue
    return None

def normalize_date(d):
    if isinstance(d, str):
        for fmt in ['%Y-%m-%d', '%B %d, %Y', '%b %d, %Y']:
             try:
                 return datetime.strptime(d, fmt)
             except:
                 continue
    return d

def get_xai_news_list(start_date, end_date=None):
    """
    Scrapes article list from https://x.ai/news
    """
    url = "https://x.ai/blog"
    # Note: user said /news but website is mostly /blog. 
    # Actually x.ai/new redirects to x.ai/blog often, but let's check both or stick to what is visible.
    # Links look like /blog/grok-2. Let's start with /blog as main index.
    
    try:
        start_dt = normalize_date(start_date)
        end_dt = normalize_date(end_date) if end_date else datetime.now()
        if start_dt: start_dt = start_dt.replace(hour=0, minute=0, second=0)
        if end_dt: end_dt = end_dt.replace(hour=23, minute=59, second=59)
        
        print(f"Fetching xAI News List... Range: {start_dt} ~ {end_dt}")
        
        response = requests.get(url, impersonate="chrome", timeout=30)
        if response.status_code != 200:
             # Fallback to /news if /blog fails
             response = requests.get("https://x.ai/news", impersonate="chrome", timeout=30)
             if response.status_code != 200:
                 print(f"Failed to access list page: {response.status_code}")
                 return []
            
        soup = BeautifulSoup(response.content, 'html.parser')
        articles = []
        
        # Strategy: xAI links start with /blog or /news
        all_links = soup.find_all('a', href=True)
        
        for link in all_links:
            href = link['href']
            if not (href.startswith('/blog/') or href.startswith('/news/')):
                continue
                
            full_url = f"https://x.ai{href}"
            
            # Date extraction from list is tricky on x.ai
            # Often it's in a sibling div or child div.
            # Heuristic: Find nearest date-like text in parent container
            found_date = None
            found_date_str = ""
            
            # Check parent content for date
            container = link
            for _ in range(3): # traverse up a few levels
                if not container: break
                text = container.get_text("\n")
                # Try to find date regex in text block
                match = re.search(r'([A-Z][a-z]+ \d{1,2}, \d{4})', text)
                if match:
                    d_can = parse_xai_date(match.group(1))
                    if d_can:
                        found_date = d_can
                        found_date_str = match.group(1)
                        break
                container = container.parent
            
            title = link.get_text(strip=True)
            if not title or len(title) < 5:
                # Maybe title is inside h3/h2 inside link
                h = link.find(['h1', 'h2', 'h3', 'h4'])
                if h: title = h.get_text(strip=True)
            
            if found_date:
                if start_dt <= found_date <= end_dt:
                    if not any(a['url'] == full_url for a in articles):
                        articles.append({
                            'url': full_url,
                            'date': found_date_str,
                            'dt': found_date,
                            'title': title
                        })
            
        print(f"Found {len(articles)} matching articles in list.")
        return articles
        
    except Exception as e:
        print(f"Error in get_xai_news_list: {e}")
        return []

def scrape_xai_article_detail(url, list_info):
    """
    Scrapes detail of a single xAI article.
    """
    try:
        response = requests.get(url, impersonate="chrome", timeout=30)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Main Content
        # Usually in div.prose or similar
        content_div = soup.find('div', class_=lambda c: c and 'prose' in c)
        if not content_div: 
             content_div = soup.find('article')
        
        if not content_div:
             # Fallback to main
             content_div = soup.find('main')
            
        if content_div:
            for tag in content_div(['script', 'style', 'button', 'nav', 'footer', 'header', 'form']):
                tag.decompose()
            raw_content = content_div.get_text(separator='\n', strip=True)
        else:
            raw_content = ""
            
        # Title extraction from Page (often better than list)
        h1 = soup.find('h1')
        page_title = h1.get_text(strip=True) if h1 else list_info['title']
        
        return {
            "raw_content": raw_content,
            "source_url": url,
            "date": list_info['date'],
            "raw_title": page_title,
            "source_name": "xAI"
        }

    except Exception as e:
        print(f"Error scraping article {url}: {e}")
        return None

def run_xai_scraper(start_date, end_date=None):
    target_articles = get_xai_news_list(start_date, end_date)
    results = []
    
    for article in target_articles:
        print(f"  > Scraping: {article['title']} ({article['date']})")
        details = scrape_xai_article_detail(article['url'], article)
        if details:
             results.append(details)
        time.sleep(1)
        
    print(f"Done. Scraped {len(results)} articles.")
    return results


In [3]:
# Test xAI Scraper
if 'run_xai_scraper' in globals():
    # Example Range
    # Note: xAI updates less frequently, so set a wide range for testing
    start_date = '2025-12-20'
    end_date = '2026-01-04'
    
    print(f"Running scraper from {start_date} to {end_date}...")
    res = run_xai_scraper(start_date, end_date)
    
    print("\n[Results]")
    for r in res:
        print(r)


Running scraper from 2025-12-20 to 2026-01-04...
Fetching xAI News List... Range: 2025-12-20 00:00:00 ~ 2026-01-04 23:59:59
Found 2 matching articles in list.
  > Scraping: Grok Collections API (December 22, 2025)
  > Scraping: Supporting the DOW's mission with AI (December 22, 2025)
Done. Scraped 2 articles.

[Results]
{'raw_content': 'Today, we\'re excited to announce Collections API. With Collections, you can upload and search through entire datasets.\nFrom PDFs and Excel sheets to entire codebases, you can upload your files into a knowledge base that supports precise and fast search.\nThis allows developers to build RAG applications without the headache of managing indexing and retrieval infrastructure.\nTo help you get started, we\'re making file indexing and storage free for the first week*, with retrieval priced at a flat rate of $2.50 per 1,000 searches.\nIndexing\nPowerful document understanding\n: We use OCR and layout-aware parsing to extract text while preserving structure 

In [4]:
r

{'raw_content': 'xAI selected by the US Department of War to power Enterprise AI and Mission Systems\nxAI is proud to announce that its suite of Frontier AI systems specially designed for United States Military and National Security users, was selected by the U.S. Department of War (DOW) for Frontier AI use cases as a part of DOW’s GenAI.Mil suite. GenAI.mil will enable 3 million military and civilian employees of the DOW to access xAI technologies at DOW Impact Level 5 (DOW IL5).\nThis new partnership between xAI and the DoW’s Chief Digital and Artificial Intelligence Office (CDAO) brings xAI’s Frontier AI systems, powered by the Grok family of models, to both Enterprise AI and critical mission use cases.\nxAI for Government is an AI platform designed for Enterprise use cases across federal, state, and local government agencies in the United States. The platform combines access to xAI’s industry-leading AI models, agentic tools, research platform, and API. xAI Government users within 

In [ ]:
import sys
sys.path.insert(0, '../agents')
from xAI_news_agent import XAINewsAgent
agent = XAINewsAgent(start_date="2026-01-01", end_date="2026-01-30")
results = agent.run()
for item in results:
    print(f"[{item['date']}] {item['raw_title']}")

Starting xAI News Scraper (2026-01-25 ~ 2026-01-30)...
Fetching xAI News List... Range: 2026-01-25 00:00:00 ~ 2026-01-30 23:59:59
Found 0 matching articles in list.
Done. Scraped 0 articles.
